In [1]:
import os
import sys
import json

import pickle
import logging
import argparse
import json_repair
import pandas as pd


# Load env from ../.env
from tqdm import tqdm
from pathlib import Path
from typing import Optional
from pydantic import BaseModel
from dotenv import load_dotenv
from collections import defaultdict

from vllm import LLM, SamplingParams
from vllm.sampling_params import GuidedDecodingParams

/Users/davidlabarbera/Documents/gDrive/Lavoro/PostDoc/Projects/psyllm/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


INFO 06-03 17:09:01 [importing.py:16] Triton not installed or not compatible; certain GPU-related functions will not be available.
WARNING 06-03 17:09:01 [importing.py:28] Triton is not installed. Using dummy decorators. Install it via `pip install triton` to enable kernel compilation.
INFO 06-03 17:09:02 [__init__.py:243] Automatically detected platform cpu.


In [2]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    handlers=[
        logging.StreamHandler(),  # Log to console
    ]
)
log = logging.getLogger(__name__)

In [3]:
def find_project_root(marker_files=["pyproject.toml", ".git", "config.json"]) -> Path:
    try:
        # Works in scripts
        current = Path(__file__).resolve().parent
    except NameError:
        # Works in notebooks or interactive sessions
        current = Path.cwd()
    
    for parent in [current] + list(current.parents):
        if any((parent / marker).exists() for marker in marker_files):
            return parent

    raise RuntimeError("Project root not found.")

In [4]:
def get_data(data_path: str, dataset: str) -> Optional[pd.DataFrame]:
    full_path = os.path.join(f"{data_path}", f"{dataset}.json")
    
    try:
        with open(full_path, "r", encoding="utf-8") as f:
            data = json.load(f)
        
        if "personas" not in data:
            log.error(f"'personas' field not found in JSON at {full_path}")
            return None

        personas_data = data["personas"]
        if not isinstance(personas_data, list):
            log.error(f"'personas' field is not a list in {full_path}")
            return None

        log.info(f"Loaded personas data from {full_path}")
        return personas_data

    except FileNotFoundError:
        log.error(f"File not found at {full_path}")
        return None
    except json.JSONDecodeError:
        log.error(f"Failed to decode JSON from {full_path}")
        return None
    except Exception as e:
        log.error(f"Unexpected error while reading {full_path}: {e}")
        return None

In [5]:
def build_llm(model_id: str, cache_path: Optional[str]) -> Optional[LLM]:
    try:
        if cache_path and os.path.isdir(cache_path):
            download_dir = cache_path
        else:
            download_dir = os.getenv("HF_HOME", os.path.expanduser("~/.cache/huggingface"))
            os.makedirs(download_dir, exist_ok=True)  # ensure it exists

        cuda_devices = os.environ.get("CUDA_VISIBLE_DEVICES", "")
        n_gpus = 1 if not cuda_devices else cuda_devices.count(",") + 1
        logging.info(f"Using {n_gpus} GPUs (CUDA_VISIBLE_DEVICES={cuda_devices})")
        logging.info(f"Download dir: {download_dir}")

        return LLM(
            model=model_id,
            tokenizer_mode="auto",
            trust_remote_code=True,
            enable_prefix_caching=True,
            max_model_len=1024,
            download_dir=download_dir,
            tensor_parallel_size=n_gpus
        )
    except Exception as e:
        logging.error(f"Failed to load model: {e}")
        return None

In [6]:
project_root = find_project_root()
env_path = os.path.join(project_root, "config", ".env")

load_dotenv(dotenv_path=env_path)

True

In [7]:
model_id   = os.getenv("model_id")
#cache_path = os.getenv("cache_path")
dataset    = get_data(data_path=os.path.join(project_root, "data/source"), dataset="test")

2025-06-03 17:09:04,126 - INFO - Loaded personas data from /Users/davidlabarbera/Documents/gDrive/Lavoro/PostDoc/Projects/psyllm/data/source/test.json


In [8]:
llm = build_llm(model_id=model_id, cache_path=None)

2025-06-03 17:09:04,141 - INFO - Using 1 GPUs (CUDA_VISIBLE_DEVICES=)
2025-06-03 17:09:04,142 - INFO - Download dir: /Users/davidlabarbera/.cache/huggingface


INFO 06-03 17:09:04 [__init__.py:31] Available plugins for group vllm.general_plugins:
INFO 06-03 17:09:04 [__init__.py:33] - lora_filesystem_resolver -> vllm.plugins.lora_resolvers.filesystem_resolver:register_filesystem_resolver
INFO 06-03 17:09:04 [__init__.py:36] All plugins in this group will be loaded. Set `VLLM_PLUGINS` to control which plugins to load.
INFO 06-03 17:09:07 [config.py:3131] Downcasting torch.float32 to torch.float16.
INFO 06-03 17:09:21 [config.py:793] This model supports multiple tasks: {'embed', 'classify', 'generate', 'score', 'reward'}. Defaulting to 'generate'.
WARNING 06-03 17:09:21 [arg_utils.py:1583] device type=cpu is not supported by the V1 Engine. Falling back to V0. 
INFO 06-03 17:09:21 [config.py:1909] Disabled the custom all-reduce kernel because it is not supported on current platform.
WARNING 06-03 17:09:21 [config.py:1543] Possibly too large swap space. 4.00 GiB out of the 8.00 GiB total CPU memory is allocated for the swap space.
WARNING 06-03 1

2025-06-03 17:09:29,967 - ERROR - Failed to load model: GPTNeoForCausalLM has no vLLM implementation and the Transformers implementation is not compatible with vLLM. Try setting VLLM_USE_V1=0.
